## Getting sequences from NCBI
### Source: Hollowell 2016b supplementary file S1

In [13]:
from Bio import Entrez, SeqIO

Entrez.email = "arahm010@ucr.edu"

def get_sequences(ids):
    fetchedSeq = []
    for seq_id in ids:
        handle = Entrez.efetch(db="nucleotide", id=seq_id, rettype="fasta", retmode="text")
        record = handle.read()
        fetchedSeq.append(record)
    return fetchedSeq


In [6]:
def make_id_from_txt(textFile):
    idList = []
    with open(textFile) as f:
        data = f.readlines()
    for i in data:
        idList.append(i.strip())
    return idList
        

In [15]:
len(make_id_from_txt('nifD.Hollowell.txt'))

349

# Remove fasta description for Hollowell seqs

In [394]:
def removeFastaDesc(fastaFile):
    seqRecords = []

    for seq in SeqIO.parse(fastaFile, 'fasta'):
        desc = str(seq.description)
        # Fourth word of Hollowell's fasta description contains 
        # isolate id
        desc = desc.split()[3]
        seq.id = desc
        seq.description = ''
        seqRecords.append(seq)

    print(len(seqRecords))
    SeqIO.write(seqRecords, fastaFile+'clean.fasta', 'fasta')

In [395]:
removeFastaDesc('Hollowell_dataset/Hollowell_2016b_CHR/dnaK.Hollowell.fasta')
removeFastaDesc('Hollowell_dataset/Hollowell_2016b_CHR/ITS.Hollowell.fasta')
removeFastaDesc('Hollowell_dataset/Hollowell_2016b_CHR/recA.Hollowell.fasta')
removeFastaDesc('Hollowell_dataset/Hollowell_2016b_CHR/glnII.Hollowell.fasta')

357
319
358
359


In [37]:
SeqIO.write(seqRecords, 'Hollowell_2016b_SYM/nifD.hollowell.fasta', 'fasta')

349

# Correct          dnaK hollowell misrepresentation
## remove repeat 

In [326]:
seq_record = {}
for seq in SeqIO.parse('CHR/dnaK_combined.fasta', 'fasta'):
    seqDesc = str(seq.description)
    if len(seqDesc.split()) != 1:
        seq.id = seqDesc.split()[3]
        seq.description = ''
        
    if seq.id not in seq_record.keys():
        seq_record[seq.id] = seq

len(seq_record.keys())

SeqIO.write(seq_record.values(), 'CHR/dnaK_combined.fasta.out', 'fasta')

534

# Correct SYM 370 dataset fasta description 

In [66]:
def updateFastaDesc(fastaFile):
    records = []
    for seq in SeqIO.parse(fastaFile, 'fasta'):
        if '|' in seq.description:
            seq.id = str(seq.description).split('|')[1]
        seq.description = ''
        records.append(seq)
        
    SeqIO.write(records, fastaFile + '.out', 'fasta')
            

In [67]:
updateFastaDesc('SYM/nifD.combined.fasta')

In [70]:
updateFastaDesc('SYM/nodA.combined.fasta')

In [73]:
updateFastaDesc('SYM/nodZ.combined.fasta')

In [74]:
updateFastaDesc('SYM/nodlL.combined.fasta')

# Remove feature id from BLASTn extracted hit sequences 

In [2]:
# input: fasta file after BLAST
# check seqID for presence of _
# if present, check splitted first part with meta file, if matched, use the splitted first part

import pandas as pd
from Bio import SeqIO

def checkID(seqID, metaSeqIDlist):
    '''for removing seq feature in blast hit'''
    if seqID not in metaSeqIDlist:
        if '_' in seqID:
            j = seqID.split('_')
            if len(j) == 2:
                k = j[0]
            else: 
                k = '_'.join(j[:-1])
            if k in metaSeqID:
                return k
            else:
                print(k, 'not in meta list')
            
        else:
            print('Not found in meta:', seqID)
    else:
        return seqID

def updateSeqID(fastaFile, metaSeqID):
    '''rewrite fasta seq ID to remove feature if present'''
    idDict = {}
    for record in SeqIO.parse(fastaFile, 'fasta'):
        record.id = checkID(record.id, metaSeqID)
        record.name = record.id
        record.description = ''
        if record.id != None:
            idDict[record.id] = record

    
    SeqIO.write(list(idDict.values()), fastaFile+'.filtered.fasta', 'fasta')
    

# load meta file
meta726 = pd.read_csv('../Metadata_Hollo.Geno.726.csv', encoding = 'latin1')
metaSeqID = list(meta726['Strain'])

#len(metaSeqID)


# Blast output seqID

updateSeqID('final/chr/ITS.370.fasta', metaSeqID)
updateSeqID('final/chr/dnaK.370.fasta', metaSeqID)
updateSeqID('final/chr/glnII.370.fasta', metaSeqID)
updateSeqID('final/chr/recA.370.fasta', metaSeqID)





In [3]:
updateSeqID('final/sym/nifD.370.fasta', metaSeqID)
updateSeqID('final/sym/nodA.370.fasta', metaSeqID)
updateSeqID('final/sym/nodL.370.fasta', metaSeqID)
updateSeqID('final/sym/nodZ.370.fasta', metaSeqID)

# Dataframe containing all (CHR+SYM) 

In [517]:
import pandas as pd

In [518]:
def getSeqID(fastaFile):
    idList = []
    for record in SeqIO.parse(fastaFile, 'fasta'):
        seqID = record.description
        idList.append(seqID)
    return idList

def addInMatrix(idList, column, matrix):
    for i in idList:
        if i in matrix.keys():
            matrix[i][column] = '+'
        else:
            matrix[i] = {column:'+'}
    return matrix

In [519]:

chrITS = getSeqID('final/chr/ITS.combined.fasta')
chrRecA = getSeqID('final/chr/recA.combined.fasta')
chrDnaK = getSeqID('final/chr/dnaK.combined.fasta')
chrGlnII = getSeqID('final/chr/glnII.combined.fasta')

print(len(chrITS), len(chrRecA), len(chrDnaK), len(chrGlnII))

562 703 634 697


In [520]:
# Angela's CHR dataset
chrITS = getSeqID('final/chr_angela//ITS.combined.fasta')
chrRecA = getSeqID('final/chr_angela/recA.combined.fasta')
chrDnaK = getSeqID('final/chr_angela//dnaK.combined.fasta')
chrGlnII = getSeqID('final/chr_angela/glnII.combined.fasta')

print(len(chrITS), len(chrRecA), len(chrDnaK), len(chrGlnII))

573 714 645 708


In [521]:
# initiate dicttionary
matrix = {}

for i in chrITS:
        matrix[i] = {'ITS':'+'}
        
len(matrix.keys())

#matrix

538

In [522]:
matrix = addInMatrix(chrRecA, 'RecA', matrix)

In [523]:

matrix = addInMatrix(chrDnaK, 'DnaK', matrix)

In [524]:
matrix = addInMatrix(chrGlnII, 'GlnII', matrix)

In [525]:
len(matrix)

680

In [526]:
symNifD = getSeqID('final/sym/nifD.combined.fasta')
#symNifD.extend(getSeqID('SYM/Hollowell_2016b_SYM/nifD.hollowell.fasta'))

symNodA = getSeqID('final/sym/nodA.combined.fasta')
#symNodA.extend(getSeqID('SYM/Hollowell_2016b_SYM/nodA.hollowell.fasta'))

symNodL = getSeqID('final/sym/nodL.combined.fasta')
#symNodL.extend(getSeqID('SYM/Hollowell_2016b_SYM/nodL.hollowell.fasta'))

symNodZ = getSeqID('final/sym/nodZ.combined.fasta')
#symNodZ.extend(getSeqID('SYM/Hollowell_2016b_SYM/nodZ.hollowell.fasta'))


In [527]:

len(symNifD), len(symNodL), len(symNodZ), len(symNodA)

(650, 596, 593, 593)

In [528]:
matrix = addInMatrix(symNifD, 'NifD', matrix)
matrix = addInMatrix(symNodA, 'NodA', matrix)
matrix = addInMatrix(symNodL, 'NodL', matrix)
matrix = addInMatrix(symNodZ, 'NodZ', matrix)

In [529]:
matrix = pd.DataFrame(matrix)

In [530]:
matrix = matrix.T

In [531]:
matrix.to_csv('final/epidemic_genotype_matrix.angela.chr.csv', header=True)

In [532]:
len(matrix['ITS'] == '+')

680

In [469]:
matrix.dropna()

,ITS,RecA,DnaK,GlnII,NifD,NodA,NodL,NodZ
CW9,+,+,+,+,+,+,+,+
CW7,+,+,+,+,+,+,+,+
CW4,+,+,+,+,+,+,+,+
19AcS5.6.1,+,+,+,+,+,+,+,+
19AcS5.2.1,+,+,+,+,+,+,+,+
...,...,...,...,...,...,...,...,...
12LoS14_2,+,+,+,+,+,+,+,+
13LoS78_1,+,+,+,+,+,+,+,+
inoc2,+,+,+,+,+,+,+,+
CW1,+,+,+,+,+,+,+,+


# Remove non-brady seq-id

In [533]:
meta = pd.read_csv('../Metadata_Hollo.Geno.726.csv', encoding = 'latin1')

In [4]:
meta.loc[meta['Rhizobial_lineage'] == 'Bradyrhizobium', 'Strain']

In [535]:
# List containing all Bradyrhizobium isolates 
bradyList = meta.loc[meta['Rhizobial_lineage'] == 'Bradyrhizobium', 'Strain']
mesoList = meta.loc[meta['Rhizobial_lineage'] == 'Mesorhizobium', 'Strain']

In [536]:
counter = 0
bradyIndex = []
mesoIndex = []
nonIndex = []

for i in list(matrix.index):
    # Not Brady
    if i not in list(bradyList):
        # Look for mesorhizobium
        if i in list(mesoList):
            mesoIndex.append(i)
        else:
            nonIndex.append(i)
    else:
        bradyIndex.append(i)
        

print(len(bradyIndex), len(mesoIndex), len(nonIndex))

575 103 2


In [537]:
# get brady isolate numbers
len(set(bradyList))

# numbers of brady isolates common in Hollowell and Lorena
counter = {}
n = 0
for i in bradyList:
    if i not in counter.keys():
        counter[i] = 1
    else:
        counter[i] += 1
        n+=1
        #print(i)

print(n)



45


In [538]:
notInBradyIndex = [i for i in bradyList if i not in bradyIndex]
notInBradyIndex

['19AcS142.1.1']

In [539]:
bradyMatrix = matrix[matrix.index.isin(bradyIndex)]
mesoMatrix = matrix[matrix.index.isin(mesoIndex)]
nonMatrix = matrix[matrix.index.isin(nonIndex)]

bradyMatrix

In [ ]:
bradyMatrix.dropna()

In [515]:
bradyMatrix.to_csv('brady_matrix.allCHR.csv', header=True)
#mesoMatrix.to_csv('meso_matrix.580.csv', header=True)
#nonMatrix.to_csv('non_matrix.580.csv', header=True)


In [545]:
bradyMatrix.dropna()

,ITS,RecA,DnaK,GlnII,NifD,NodA,NodL,NodZ
CW9,+,+,+,+,+,+,+,+
CW7,+,+,+,+,+,+,+,+
CW4,+,+,+,+,+,+,+,+
19AcS5.6.1,+,+,+,+,+,+,+,+
19AcS5.2.1,+,+,+,+,+,+,+,+
...,...,...,...,...,...,...,...,...
12LoS14_2,+,+,+,+,+,+,+,+
13LoS78_1,+,+,+,+,+,+,+,+
inoc2,+,+,+,+,+,+,+,+
CW1,+,+,+,+,+,+,+,+


# Heatmap for bradyMatrix

In [482]:
import seaborn as sns

#sns.heatmap(bradyMatrix)

## Check nonMatrix further to see if Brady can be found

In [204]:
finalNonIndex = []
extraBradyIndex = []



for i in nonIndex:
    if '_' in i:
        splited = i.split('_')
        if len(splited) == 3:
            j = '_'.join(splited[:2])
            if j not in list(bradyList):
                finalNonIndex.append(j)
                #nonIndex.remove(i)
            else:
                extraBradyIndex.append(j)
                #nonIndex.remove(i)
        else:
            finalNonIndex.append(i)
            #nonIndex.remove(i)
    else:
        finalNonIndex.append(i)
        #nonIndex.remove(i)

In [5]:
#print(len(finalNonIndex), len(extraBradyIndex), len(nonIndex))

#for i in finalNonIndex:
#    print(i)

363


In [206]:
# Brady which was found after some more edit on ID character (leaving the feature id)
extraBrady = matrix[matrix.index.isin(extraBradyIndex)]
#extraBrady.to_csv('extraBrady_matrix.csv', header=True)

# all of them already in the Brady Matrix

In [190]:
def returnNotMatches(a, b):
    return [[x for x in a if x not in b], [x for x in b if x not in a]]

def returnMatches(a, b):
    return [[x for x in a if x in b], [x for x in b if x in a]]

In [130]:
#list(bradyMatrix.index)

In [230]:
# ID could not found on the meta file
#finalNonIndex

# Check combined (370+Hollowell) SYM fasta ids exclusivity in Brady meta matrix

In [209]:
from Bio import SeqIO


# get seq records in a list
# filter seqID present in brady matrix or not
# matrix is panda dataframe

def getSeqRecord(seqFile, matrix):
    #if type(matrix) != list:
    #    matrix = list(matrix.index)
    seqRecords = []
    for seq in SeqIO.parse(seqFile, 'fasta'):
        if seq.id in matrix:
            seqRecords.append(seq)
    return seqRecords
    
    

In [268]:
# Filter and write output for bradyrhizobium SYM locuses

filterSeqs = getSeqRecord('SYM/combined_370_Hollowell/nodA.all.fasta', bradyMatrix)
SeqIO.write(filterSeqs, 'SYM/combined_370_Hollowell/nodA.brady.fasta', 'fasta')

525

# Make sure 8 genes present for all isolates

In [321]:
# need an updated bradyMatrix
chrLocus = bradyMatrix[['ITS', 'RecA', 'DnaK', 'GlnII']]
symLocus = bradyMatrix[['NifD', 'NodA', 'NodL', 'NodZ']]


In [437]:
allLoci = list(bradyMatrix.index)
len(allLoci)

559

In [337]:
bradyMatrix = bradyMatrix.dropna()

In [207]:
allLoci[:5]

['19AcS295.2.1', '19AcS136.2.1', '19AcAm129.2.1', '19AcAm211.3.1', '09LoS35_1']

In [347]:
seqAllLoci = getSeqRecord('SYM/combined_370_Hollowell/nodL.brady.fasta', allLoci)
SeqIO.write(seqAllLoci, '8_genes/nodL.8genes.3.fasta', 'fasta')

481

In [351]:
seqAllLoci = getSeqRecord('CHR/glnII.brady.580.fasta', allLoci)
SeqIO.write(seqAllLoci, '8_genes/glnII.8genes.3.fasta', 'fasta')

544

# Keep only unique SeqID
### Remove the smallest in size

In [332]:
from Bio import SeqIO

def removeRepeat(fastaFile):
    seqRecords = {}
    for seq in SeqIO.parse(fastaFile, 'fasta'):
        if seq.id not in seqRecords.keys():
            seqRecords[seq.id] = seq
        else:
            lenA = len(str(seqRecords[seq.id].seq))
            lenB = len(str(seq.seq))
            if lenB > lenA:
                seqRecords[seq.id] = seq
    SeqIO.write(seqRecords.values(), fastaFile + '.unique', 'fasta')
        



In [352]:
removeRepeat('8_genes/dnaK.8genes.3.fasta')
removeRepeat('8_genes/glnII.8genes.3.fasta')
removeRepeat('8_genes/ITS.8genes.3.fasta')
removeRepeat('8_genes/recA.8genes.3.fasta')

removeRepeat('8_genes/nifD.8genes.3.fasta')
removeRepeat('8_genes/nodZ.8genes.3.fasta')
removeRepeat('8_genes/nodL.8genes.3.fasta')
removeRepeat('8_genes/nodA.8genes.3.fasta')

# Check BLAST extracted datset to find missing sequences

In [285]:
chrITS = getSeqID('CHR/ITS.brady.580.fasta')
chrRecA = getSeqID('CHR/recA.brady.580.fasta')
chrDnaK = getSeqID('CHR/dnaK.brady.580.fasta')
chrGlnII = getSeqID('CHR/glnII.brady.580.fasta')

bradyID = list(meta.loc[meta['Rhizobial_lineage'] == 'Bradyrhizobium']['Strain'])

def findMissing(bradyID, targetID):
    missing = []
    for i in bradyID:
        if i not in targetID:
            missing.append(i)
    return missing

In [287]:
missingITS = findMissing(bradyID, chrITS)
missingRecA = findMissing(bradyID, chrRecA)
missingDnaK = findMissing(bradyID, chrDnaK)
missingGlnII = findMissing(bradyID, chrGlnII)

In [291]:
symNifD = getSeqID('SYM/nifD.combined.fasta.outid.fasta')
symNifD.extend(getSeqID('SYM/Hollowell_2016b_SYM/nifD.hollowell.fasta'))

symNodA = getSeqID('SYM/nodA.combined.fasta.outid.fasta')
symNodA.extend(getSeqID('SYM/Hollowell_2016b_SYM/nodA.hollowell.fasta'))

symNodL = getSeqID('SYM/nodlL.combined.fasta.outid.fasta')
symNodL.extend(getSeqID('SYM/Hollowell_2016b_SYM/nodL.hollowell.fasta'))

symNodZ = getSeqID('SYM/nodZ.combined.fasta.outid.fasta')
symNodZ.extend(getSeqID('SYM/Hollowell_2016b_SYM/nodZ.hollowell.fasta'))

In [292]:
missingNifD = findMissing(bradyID, symNifD)
missingNodA = findMissing(bradyID, symNodA)
missingNodL = findMissing(bradyID, symNodL)
missingNodZ = findMissing(bradyID, symNodZ)

In [307]:
missingList = {
    'ITS':missingITS,
    'RecA':missingRecA,
    'DnaK':missingDnaK,
    'GlnII':missingGlnII,
    'NifD':missingNifD,
    'NodA':missingNodA,
    'NodL':missingNodL,
    'NodZ':missingNodZ
}

missingDict = {}
# {
# 1:{its:+, recA:+, dnaK:+}
# }


for k in missingList.keys(): # k for gene name keys
    for i in missingList[k]: # i for index of seq
        if i not in missingDict.keys():
            missingDict[i] = {k:'missing'}
        else:
            missingDict[i][k] = 'missing'

In [310]:
missingMat = pd.DataFrame.from_dict(missingDict)
missingMat = missingMat.transpose()
missingMat.to_csv('missing_matrix.csv', header=True)

In [311]:
meta

,Source,Strain,Site,Latitude,Longitude,Host,Rhizobial_lineage,ANI_species
0,Hollowell,05LoS1_2,Bodega Marine Reserve,38.319300,-123.063733,A. strigosus,Bradyrhizobium,NaN
1,Hollowell,05LoS1_3,Bodega Marine Reserve,38.319300,-123.063733,A. strigosus,Bradyrhizobium,NaN
2,Hollowell,05LoS1_4,Bodega Marine Reserve,38.319300,-123.063733,A. strigosus,Bradyrhizobium,NaN
3,Hollowell,05LoS1_5,Bodega Marine Reserve,38.319300,-123.063733,A. strigosus,Bradyrhizobium,NaN
4,Hollowell,05LoS1_6,Bodega Marine Reserve,38.319300,-123.063733,A. strigosus,Bradyrhizobium,NaN
...,...,...,...,...,...,...,...,...
721,Genomes,13LoS58_2,UC Riverside Hills,33.965933,-117.323000,A. strigosus,Bradyrhizobium,49.0
722,Genomes,CW1,UC Riverside Hills,33.965933,-117.323000,A. strigosus,Bradyrhizobium,24.0
723,Genomes,CW4,UC Riverside Hills,33.965933,-117.323000,A. strigosus,Bradyrhizobium,24.0
724,Genomes,13LoS100_1,Whitewater Preserve,33.979720,-116.654700,A. strigosus,Bradyrhizobium,31.0


# Create final Bradyrhizobium dataset by filtering

In [548]:
metaBrady = meta[meta['Rhizobial_lineage'] == 'Bradyrhizobium']

In [558]:
metaBrady

,Source,Strain,Site,Latitude,Longitude,Host,Rhizobial_lineage,ANI_species
0,Hollowell,05LoS1_2,Bodega Marine Reserve,38.319300,-123.063733,A. strigosus,Bradyrhizobium,NaN
1,Hollowell,05LoS1_3,Bodega Marine Reserve,38.319300,-123.063733,A. strigosus,Bradyrhizobium,NaN
2,Hollowell,05LoS1_4,Bodega Marine Reserve,38.319300,-123.063733,A. strigosus,Bradyrhizobium,NaN
3,Hollowell,05LoS1_5,Bodega Marine Reserve,38.319300,-123.063733,A. strigosus,Bradyrhizobium,NaN
4,Hollowell,05LoS1_6,Bodega Marine Reserve,38.319300,-123.063733,A. strigosus,Bradyrhizobium,NaN
...,...,...,...,...,...,...,...,...
721,Genomes,13LoS58_2,UC Riverside Hills,33.965933,-117.323000,A. strigosus,Bradyrhizobium,49.0
722,Genomes,CW1,UC Riverside Hills,33.965933,-117.323000,A. strigosus,Bradyrhizobium,24.0
723,Genomes,CW4,UC Riverside Hills,33.965933,-117.323000,A. strigosus,Bradyrhizobium,24.0
724,Genomes,13LoS100_1,Whitewater Preserve,33.979720,-116.654700,A. strigosus,Bradyrhizobium,31.0


In [560]:
# Function to filter fasta file by membership in a list
from Bio import SeqIO

def filterByList(fasFile, idList):
    filteredList = []
    for seq_record in SeqIO.parse(fasFile, 'fasta'):
        if seq_record.id in idList:
            filteredList.append(seq_record)
    SeqIO.write(filteredList, fasFile+'.filtered.fasta', 'fasta')

In [554]:
filterByList('Bradyrhizobium/Dataset/Complete/sym/nifD.combined.fasta', list(metaBrady['Strain']))
filterByList('Bradyrhizobium/Dataset/Complete/sym/nodZ.combined.fasta', list(metaBrady['Strain']))
filterByList('Bradyrhizobium/Dataset/Complete/sym/nodL.combined.fasta', list(metaBrady['Strain']))
filterByList('Bradyrhizobium/Dataset/Complete/sym/nodA.combined.fasta', list(metaBrady['Strain']))

In [557]:
filterByList('Bradyrhizobium/Dataset/Complete/chr/dnaK.combined.fasta', list(metaBrady['Strain']))
filterByList('Bradyrhizobium/Dataset/Complete/chr/glnII.combined.fasta', list(metaBrady['Strain']))
filterByList('Bradyrhizobium/Dataset/Complete/chr/recA.combined.fasta', list(metaBrady['Strain']))
filterByList('Bradyrhizobium/Dataset/Complete/chr/ITS.combined.fasta', list(metaBrady['Strain']))

# Create Hollowell dataset by filtering

In [559]:
metaHoll = meta[meta['Source'] == 'Hollowell']

In [562]:
filterByList('Bradyrhizobium/Dataset/Hollowell/sym/nifD.brady.fasta', list(metaHoll['Strain']))
filterByList('Bradyrhizobium/Dataset/Hollowell/sym/nodZ.brady.fasta', list(metaHoll['Strain']))
filterByList('Bradyrhizobium/Dataset/Hollowell/sym/nodL.brady.fasta', list(metaHoll['Strain']))
filterByList('Bradyrhizobium/Dataset/Hollowell/sym/nodA.brady.fasta', list(metaHoll['Strain']))


In [563]:
filterByList('Bradyrhizobium/Dataset/Hollowell/chr/dnaK.bradyrhizobium.fasta', list(metaHoll['Strain']))
filterByList('Bradyrhizobium/Dataset/Hollowell/chr/glnII.bradyrhizobium.fasta', list(metaHoll['Strain']))
filterByList('Bradyrhizobium/Dataset/Hollowell/chr/recA.bradyrhizobium.fasta', list(metaHoll['Strain']))
filterByList('Bradyrhizobium/Dataset/Hollowell/chr/ITS.bradyrhizobium.fasta', list(metaHoll['Strain']))


# Remove duplicated sequences in fasta file

In [36]:
from Bio import SeqIO

def removeDuplicates(fasFile):
    seqHolder = {}
    for seq_record in SeqIO.parse(fasFile, 'fasta'):
        if seq_record.id not in seqHolder.keys():
            seqHolder[seq_record.id] = seq_record
        else:
            if len(str(seq_record.seq)) > len(str(seqHolder[seq_record.id].seq)):
                seqHolder[seq_record.id] = seq_record
    SeqIO.write(seqHolder.values(), fasFile, 'fasta')

In [37]:
removeDuplicates('Bradyrhizobium/Dataset/Complete/chr/ITS.bradyrhizobium.fasta')
removeDuplicates('Bradyrhizobium/Dataset/Complete/chr/dnaK.bradyrhizobium.fasta')
removeDuplicates('Bradyrhizobium/Dataset/Complete/chr/recA.bradyrhizobium.fasta')
removeDuplicates('Bradyrhizobium/Dataset/Complete/chr/glnII.bradyrhizobium.fasta')

removeDuplicates('Bradyrhizobium/Dataset/Complete/sym/nifD.brady.fasta')
removeDuplicates('Bradyrhizobium/Dataset/Complete/sym/nodA.brady.fasta')
removeDuplicates('Bradyrhizobium/Dataset/Complete/sym/nodZ.brady.fasta')
removeDuplicates('Bradyrhizobium/Dataset/Complete/sym/nodL.brady.fasta')

# Make haplotype dictionary from Hollowell S1

In [645]:
import pandas as pd


# Get S1 from hollowell 2016b in a dataframe 
hollowell = pd.read_csv('Bradyrhizobium/Dataset/hollowell_haplotypes.csv', encoding = 'latin1')
hollowell = hollowell.set_index('Isolate')

In [788]:
#init emptyHashDict

def emptyHashDict():
    hashDict = {}
    for i in list(hollowell.index):
        if i not in hashDict.keys():
            hashDict[i] = 'NAN'
    return hashDict

def hashHaplotypes(namesFile):
    hashDict = emptyHashDict()
    with open(namesFile) as f:
        data = f.readlines()
    for i in data:
        hap, mem = i.split('\t')
        mem = mem.strip().split(',')
        hashDict[hap] = mem
        
    return(hashDict)
    

In [789]:
dnaK_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/chr/dnaK.hollowell.fasta.names')
recA_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/chr/recA.hollowell.fasta.names')
ITS_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/chr/ITS.hollowell.fasta.names')
glnII_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/chr/glnII.hollowell.fasta.names')

nifD_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/sym/nifD.hollowell.fasta.names')
nodA_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/sym/nodA.hollowell.fasta.names')
nodL_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/sym/nodL.hollowell.fasta.names')
nodZ_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/sym/nodZ.hollowell.fasta.names')

In [790]:
# Generic function to convert namefiles from Mothur output (unique.seqs cmmand) 
haplotypes = {}
haplotypes = {'dnaK':dnaK_names, 'recA':recA_names, 'ITS':ITS_names, 'glnII':glnII_names,
             'nifD':nifD_names, 'nodA':nodA_names, 'nodL':nodL_names, 'nodZ':nodZ_names}


In [791]:
haplotypes = pd.DataFrame.from_dict(haplotypes)

### Initialize hollowell-haplotype match by keys of haplotype

In [792]:



# given list of isolates id, search for hollowell in a particular loci
def initHapMatch(haplotypeID, loci):
    isolateHapMat = emptyHashDict()
    for i in haplotypeID:
        hap = hollowell.loc[i, loci]
        isolateHapMat[i] = hap
    return(isolateHapMat)
        

In [793]:
dnaK_keys = list(haplotypes[haplotypes['dnaK'] != 'NA'].index)
ITS_keys = list(haplotypes[haplotypes['ITS'] != 'NA'].index)
recA_keys = list(haplotypes[haplotypes['recA'] != 'NA'].index)
glnII_keys = list(haplotypes[haplotypes['glnII'] != 'NA'].index)

nifD_keys = list(haplotypes[haplotypes['nifD'] != 'NA'].index)
nodA_keys = list(haplotypes[haplotypes['nodA'] != 'NA'].index)
nodL_keys = list(haplotypes[haplotypes['nodL'] != 'NA'].index)
nodZ_keys = list(haplotypes[haplotypes['nodZ'] != 'NA'].index)

dnaK_init = initHapMatch(dnaK_keys, 'dnak' )
ITS_init = initHapMatch(ITS_keys, 'ITS' )
recA_init = initHapMatch(recA_keys, 'recA' )
glnII_init = initHapMatch(glnII_keys, 'glnII' )

nifD_init = initHapMatch(nifD_keys, 'nifD' )
nodA_init = initHapMatch(nodA_keys, 'nodD-A' )
nodL_init = initHapMatch(nodL_keys, 'nolL' )
nodZ_init = initHapMatch(nodZ_keys, 'nodZ' )

In [795]:
initMat = {}

initMat = {'dnaK':dnaK_init, 'ITS':ITS_init, 'recA':recA_init, 'glnII':glnII_init, 
           'nifD':nifD_init, 'nodA':nodA_init, 'nodL':nodL_init, 'nodZ':nodZ_init}
initMat = pd.DataFrame.from_dict(initMat)

initMat.head()

,dnaK,ITS,recA,glnII,nifD,nodA,nodL,nodZ
05LoS1_2,K01,I01,R01,G01,D49,N01,L38,Z28
05LoS1_3,K01,I01,R01,G01,D01,N02,L01,Z01
05LoS1_4,K01,I01,R01,G01,D01,N02,L01,Z01
05LoS1_5,K21,I33,R02,G107,D02,N02,L02,Z29
05LoS1_6,K01,I01,R01,G01,D01,N03,L01,Z01


### After initializing initMat, fill the NA cells with other haplotypes belongs to the same group as per as Mothur analysis

In [797]:
# So far we have original `haplotype` matrix and an `initMat` matrix

haplotypes.head()


,dnaK,recA,ITS,glnII,nifD,nodA,nodL,nodZ
05LoS1_2,NAN,NAN,NAN,NAN,[05LoS1_2],NAN,[05LoS1_2],[05LoS1_2]
05LoS1_3,NAN,NAN,NAN,NAN,NAN,"[05LoS1_3, 05LoS21_4, 05LoS22_11]",NAN,NAN
05LoS1_4,NAN,NAN,NAN,NAN,NAN,NAN,NAN,NAN
05LoS1_5,[05LoS1_5],NAN,[05LoS1_5],[05LoS1_5],NAN,[05LoS1_5],NAN,[05LoS1_5]
05LoS1_6,NAN,NAN,NAN,NAN,NAN,NAN,NAN,NAN


In [746]:
#initMat.head()
initMat.loc['05LoS1_5' , 'dnaK']

'K21'

In [798]:
# Search for non-NA entry in the haplotypes and update it in the initMat

def fillInitMat(loci):    
    subset = haplotypes[haplotypes[loci] != 'NAN'][loci]
    for i in subset.index:
        if len(subset[i]) > 1:
            hapID = subset[i][0]
            hap = initMat.loc[hapID, loci]
            #print(hap)
            for j in subset[i][1:]:
                initMat.loc[j, loci] = hap
            



In [801]:
fillInitMat('dnaK')
fillInitMat('recA')
fillInitMat('ITS')
fillInitMat('glnII')

fillInitMat('nifD')
fillInitMat('nodA')
fillInitMat('nodL')
fillInitMat('nodZ')

In [800]:
initMat.to_csv('Bradyrhizobium/Dataset/reproduce_Hollowell_nomenclature.csv')

### Check ambiguous cells between `initMat` and `hollowell` 

In [816]:
#initMat.head()
#hollowell.head()


# Function for comparing initMat and hollowell cells for each loci

def findDiscrep(lociA, lociB):
    for i in list(hollowell.index):
        if hollowell.loc[i, lociA] != initMat.loc[i,lociB]:
            print(i, hollowell.loc[i, lociA], initMat.loc[i, lociB])
            
            
print('dnaK:')
findDiscrep('dnak', 'dnaK')
print('ITS:')
findDiscrep('ITS', 'ITS')
print('recA:')
findDiscrep('recA', 'recA')
print('glnII:')
findDiscrep('glnII', 'glnII')
print('nifD:')
findDiscrep('nifD', 'nifD')
print('nodA:')
findDiscrep('nodD-A', 'nodA')
print('nodL:')
findDiscrep('nolL', 'nodL')
print('nodZ:')
findDiscrep('nodZ', 'nodZ')
#initMat.loc['05LoS1_2', 'dnaK']

In [809]:
initMat.head()


,dnaK,ITS,recA,glnII,nifD,nodA,nodL,nodZ
05LoS1_2,K01,I01,R01,G01,D49,N01,L38,Z28
05LoS1_3,K01,I01,R01,G01,D01,N02,L01,Z01
05LoS1_4,K01,I01,R01,G01,D01,N08,L01,Z01
05LoS1_5,K21,I33,R98,G107,D02,N02,L07,Z29
05LoS1_6,K01,I01,R01,G01,D01,N03,L01,Z01


In [810]:
hollowell.head()

,Unnamed: 0,Site,Latitude,Longitude,dnak,glnII,ITS,recA,nifD,nodD-A,nodZ,nolL,chromosome haplotype,symbiosis island haplotype,genome haplotype,CHR clade,SI lineage
Isolate,,,,,,,,,,,,,,,,,
05LoS1_2,1,Bodega Marine Reserve,38.319300¡,-123.063733¡,K01,G01,I01,R01,D49,N01,Z28,L38,K01_G01_I01_R01,D49_N01_Z28_L38,K01_G01_I01_R01_D49_N01_Z28_L38,B. canariense,4
05LoS1_3,2,Bodega Marine Reserve,38.319300¡,-123.063733¡,K01,G01,I01,R01,D01,N02,Z01,L01,K01_G01_I01_R01,D01_N02_Z01_L01,K01_G01_I01_R01_D01_N02_Z01_L01,B. canariense,4
05LoS1_4,3,Bodega Marine Reserve,38.319300¡,-123.063733¡,K01,G01,I01,R01,D01,N02,Z01,L01,K01_G01_I01_R01,D01_N02_Z01_L01,K01_G01_I01_R01_D01_N02_Z01_L01,B. canariense,4
05LoS1_5,4,Bodega Marine Reserve,38.319300¡,-123.063733¡,K21,G107,I33,R02,D02,N02,Z29,L02,K21_G107_I33_R02,D02_N02_Z29_L02,K21_G107_I33_R02_D02_N02_Z29_L02,B. canariense,4
05LoS1_6,5,Bodega Marine Reserve,38.319300¡,-123.063733¡,K01,G01,I01,R01,D01,N03,Z01,L01,K01_G01_I01_R01,D01_N03_Z01_L01,K01_G01_I01_R01_D01_N03_Z01_L01,B. canariense,4


# Matching with Hollowell Master dataset

In [26]:
import pandas as pd


# Get modified master from hollowell 2016b in a dataframe 
hollowell = pd.read_csv('Hollowell_files/Master_haplotypes.csv', encoding = 'latin1')
hollowell = hollowell.set_index('ID')

#init emptyHashDict

def emptyHashDict():
    hashDict = {}
    for i in list(hollowell.index):
        if i not in hashDict.keys():
            hashDict[i] = 'NAN'
    return hashDict

def hashHaplotypes(namesFile):
    hashDict = emptyHashDict()
    with open(namesFile) as f:
        data = f.readlines()
    for i in data:
        hap, mem = i.split('\t')
        mem = mem.strip().split(',')
        hashDict[hap] = mem
        
    return(hashDict)




# given list of isolates id, search for hollowell in a particular loci
def initHapMatch(haplotypeID, loci):
    isolateHapMat = emptyHashDict()
    for i in haplotypeID:
        hap = hollowell.loc[i, loci]
        isolateHapMat[i] = hap
    return(isolateHapMat)

# Search for non-NA entry in the haplotypes and update it in the initMat

def fillInitMat(loci):    
    subset = haplotypes[haplotypes[loci] != 'NAN'][loci]
    for i in subset.index:
        if len(subset[i]) > 1:
            hapID = subset[i][0]
            hap = initMat.loc[hapID, loci]
            #print(hap)
            for j in subset[i][1:]:
                initMat.loc[j, loci] = hap
            



In [2]:
dnaK_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/chr/dnaK.hollowell.fasta.names')
recA_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/chr/recA.hollowell.fasta.names')
ITS_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/chr/ITS.hollowell.fasta.names')
glnII_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/chr/glnII.hollowell.fasta.names')

nifD_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/sym/nifD.hollowell.fasta.names')
nodA_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/sym/nodA.hollowell.fasta.names')
nodL_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/sym/nodL.hollowell.fasta.names')
nodZ_names = hashHaplotypes('Bradyrhizobium/Dataset/Hollowell/sym/nodZ.hollowell.fasta.names')

In [16]:
# Generic function to convert namefiles from Mothur output (unique.seqs cmmand) 
haplotypes = {}
haplotypes = {'dnaK':dnaK_names, 'recA':recA_names, 'ITS':ITS_names, 'glnII':glnII_names,
             'nifD':nifD_names, 'nodA':nodA_names, 'nodL':nodL_names, 'nodZ':nodZ_names}
haplotypes = pd.DataFrame.from_dict(haplotypes)

In [17]:
haplotypes

,dnaK,recA,ITS,glnII,nifD,nodA,nodL,nodZ
12LoS20_4,NAN,[12LoS20_4],NAN,[12LoS20_4],NAN,NAN,NAN,NAN
12LoS21_22,NAN,NAN,NAN,NAN,NAN,NAN,NAN,NAN
11LoS11_3,[11LoS11_3],[11LoS11_3],[11LoS11_3],[11LoS11_3],"[11LoS11_3, 12LoS4_2]","[11LoS11_3, 12LoS4_2]","[11LoS11_3, 11LoS20_2, 12LoS4_2]",NAN
11LoS20_2,[11LoS20_2],NAN,[11LoS20_2],[11LoS20_2],[11LoS20_2],[11LoS20_2],NAN,NAN
11LoS20_4,[11LoS20_4],"[11LoS20_4, 11LoS20_10]",[11LoS20_4],[11LoS20_4],NAN,NAN,[11LoS20_4],NAN
...,...,...,...,...,...,...,...,...
13LoS120_3,NAN,NAN,NAN,NAN,NAN,NAN,NAN,NAN
13LoS120_4,NAN,NAN,NAN,NAN,NAN,NAN,NAN,NAN
13LoS120_5,NAN,NAN,NAN,NAN,NAN,NAN,NAN,NAN
13LoS120_7,NAN,NAN,NAN,NAN,NAN,NAN,NAN,NAN


In [18]:
dnaK_keys = list(haplotypes[haplotypes['dnaK'] != 'NA'].index)
ITS_keys = list(haplotypes[haplotypes['ITS'] != 'NA'].index)
recA_keys = list(haplotypes[haplotypes['recA'] != 'NA'].index)
glnII_keys = list(haplotypes[haplotypes['glnII'] != 'NA'].index)

nifD_keys = list(haplotypes[haplotypes['nifD'] != 'NA'].index)
nodA_keys = list(haplotypes[haplotypes['nodA'] != 'NA'].index)
nodL_keys = list(haplotypes[haplotypes['nodL'] != 'NA'].index)
nodZ_keys = list(haplotypes[haplotypes['nodZ'] != 'NA'].index)

In [20]:
len(dnaK_keys)

1673

In [23]:

dnaK_init = initHapMatch(dnaK_keys, 'dnaK_haplotype' )
ITS_init = initHapMatch(ITS_keys, 'ITS_haplotype' )
recA_init = initHapMatch(recA_keys, 'recA_haplotype' )
glnII_init = initHapMatch(glnII_keys, 'glnII_haplotype' )

nifD_init = initHapMatch(nifD_keys, 'nifD_haplotype' )
nodA_init = initHapMatch(nodA_keys, 'nodDA_haplotype' )
nodL_init = initHapMatch(nodL_keys, 'nolL_haplotype' )
nodZ_init = initHapMatch(nodZ_keys, 'nodZ_haplotype' )

In [25]:
initMat = {}

initMat = {'dnaK':dnaK_init, 'ITS':ITS_init, 'recA':recA_init, 'glnII':glnII_init, 
           'nifD':nifD_init, 'nodA':nodA_init, 'nodL':nodL_init, 'nodZ':nodZ_init}
initMat = pd.DataFrame.from_dict(initMat)

initMat.head()

,dnaK,ITS,recA,glnII,nifD,nodA,nodL,nodZ
12LoS20_4,K14,X,R220,G60,X,X,X,X
12LoS21_22,K20,X,R27,G58,D28,N42,X,X
11LoS11_3,K31,I89,R204,G212,D24,N21,L17,X
11LoS20_2,K34,I100,R209,G219,D76,N84,L17,X
11LoS20_4,K35,I101,R57,G19,X,X,L57,X


In [27]:
fillInitMat('dnaK')
fillInitMat('recA')
fillInitMat('ITS')
fillInitMat('glnII')

fillInitMat('nifD')
fillInitMat('nodA')
fillInitMat('nodL')
fillInitMat('nodZ')

In [28]:
initMat.head()

,dnaK,ITS,recA,glnII,nifD,nodA,nodL,nodZ
12LoS20_4,K14,X,R220,G60,X,X,X,X
12LoS21_22,K20,X,R27,G58,D28,N42,X,X
11LoS11_3,K31,I89,R204,G212,D24,N21,L17,X
11LoS20_2,K34,I100,R03,G219,D76,N84,L17,X
11LoS20_4,K35,I101,R57,G19,X,X,L57,X


In [29]:
# Check ambiguous entry between hollowell and mine

# Function for comparing initMat and hollowell cells for each loci

def findDiscrep(lociA, lociB):
    for i in list(hollowell.index):
        if hollowell.loc[i, lociA] != initMat.loc[i,lociB]:
            print(i, hollowell.loc[i, lociA], initMat.loc[i, lociB])
            
            
print('dnaK:')
findDiscrep('dnaK_haplotype', 'dnaK')
print('ITS:')
findDiscrep('ITS_haplotype', 'ITS')
print('recA:')
findDiscrep('recA_haplotype', 'recA')
print('glnII:')
findDiscrep('glnII_haplotype', 'glnII')
print('nifD:')
findDiscrep('nifD_haplotype', 'nifD')
print('nodA:')
findDiscrep('nodDA_haplotype', 'nodA')
print('nodL:')
findDiscrep('nolL_haplotype', 'nodL')
print('nodZ:')
findDiscrep('nodZ_haplotype', 'nodZ')
#initMat.loc['05LoS1_2', 'dnaK']
  

In [31]:
hollowell.head()

,site,dnaK_haplotype,dnak_accession,glnII_haplotype,glnII_accession,ITS_haplotype,ITS_accession,recA_haplotype,recA_accession,nifD_haplotype,nifD_accession,nodDA_haplotype,nodDA_accession,nodZ_haplotype,nodZ_accession,nolL_haplotype,nolL_accession
ID,,,,,,,,,,,,,,,,,
12LoS20_4,Anza_Borrego_Road,K14,KP832259,G60,KP830511,X,X,R220,KP831410,X,X,X,X,X,X,X,X
12LoS21_22,Anza_Borrego_Road,K20,KP832286,G58,KP830538,X,X,R27,KP831437,D28,KP833298,N42,KP833608,X,X,X,X
11LoS11_3,San_Dimas_Reservoir,K31,KP832096,G212,KM194974,I89,KM195391,R204,KM195870,D24,KP833143,N21,KP833451,X,X,L17,KP832453
11LoS20_2,San_Dimas_Canyon_Slope,K34,KP832137,G219,KM195025,I100,KM195442,R209,KM195921,D76,KP833178,N84,KP833487,X,X,L17,KP832494
11LoS20_4,San_Dimas_Canyon_Slope,K35,KP832139,G19,KM195027,I101,KM195444,R57,KM195923,X,X,X,X,X,X,L57,KP832496


# Keep duplicates of Hollowell and WGS dataset

In [49]:
from Bio import SeqIO

def keepDuplicates(fasFile):
    seqHolder = {}
    duplicates = []
    for seq_record in SeqIO.parse(fasFile, 'fasta'):
        if seq_record.id not in seqHolder.keys():
            seqHolder[seq_record.id] = [seq_record]
        else:
            seqHolder[seq_record.id].append(seq_record)
            duplicates.extend(seqHolder[seq_record.id])
    #print(duplicates)
    SeqIO.write(duplicates, fasFile+'duplicate.fasta', 'fasta')

In [50]:
keepDuplicates('Bradyrhizobium/Dataset/Complete/withDuplicate/CHR/ITS.combined.fasta')
keepDuplicates('Bradyrhizobium/Dataset/Complete/withDuplicate/CHR/dnaK.combined.fasta')
keepDuplicates('Bradyrhizobium/Dataset/Complete/withDuplicate/CHR/recA.combined.fasta')
keepDuplicates('Bradyrhizobium/Dataset/Complete/withDuplicate/CHR/glnII.combined.fasta')

In [51]:
keepDuplicates('Bradyrhizobium/Dataset/Complete/withDuplicate/SYM/nodL.combined.fasta')
keepDuplicates('Bradyrhizobium/Dataset/Complete/withDuplicate/SYM/nodZ.combined.fasta')
keepDuplicates('Bradyrhizobium/Dataset/Complete/withDuplicate/SYM/nodA.combined.fasta')
keepDuplicates('Bradyrhizobium/Dataset/Complete/withDuplicate/SYM/nifD.combined.fasta')


## After aligning the duplicate sequence files, check for similar id and get hamming distance

In [103]:
from Bio import SeqIO

def getSameSeqPair(fastaFile):
    pairs = {}
    for seq_record in SeqIO.parse(fastaFile, 'fasta'):
        if seq_record.id not in pairs.keys():
            pairs[seq_record.id] = [str(seq_record.seq)]
        else:
            pairs[seq_record.id].append(str(seq_record.seq))
    return pairs
            

def hamming_dist(seqPairs):
    count = 0
    if len(seqPairs[0]) == len(seqPairs[1]):
        for i in range(len(seqPairs[0])):
            if seqPairs[0][i] != seqPairs[1][i]:
                count += 1
    return count

def hammingCompareLoci(fastaFile):
    mismatch = {}
    pairs = getSameSeqPair(fastaFile)
    for k in pairs.keys():
        dist = hamming_dist(pairs[k])
        if dist != 0:
            mismatch[k] = dist
    return mismatch
    

In [104]:
glnII_mismatch = hammingCompareLoci('Bradyrhizobium/Dataset/Complete/withDuplicate/CHR/glnII.combined.duplicate.fasta.aln')
recA_mismatch = hammingCompareLoci('Bradyrhizobium/Dataset/Complete/withDuplicate/CHR/recA.combined.duplicate.fasta.aln')
ITS_mismatch = hammingCompareLoci('Bradyrhizobium/Dataset/Complete/withDuplicate/CHR/ITS.combined.duplicate.fasta.aln')
dnaK_mismatch = hammingCompareLoci('Bradyrhizobium/Dataset/Complete/withDuplicate/CHR/dnaK.combined.duplicate.fasta.aln')

In [114]:
nifD_mismatch = hammingCompareLoci('Bradyrhizobium/Dataset/Complete/withDuplicate/SYM/nifD.combined.duplicate.fasta.aln')
nodA_mismatch = hammingCompareLoci('Bradyrhizobium/Dataset/Complete/withDuplicate/SYM/nodA.combined.duplicate.fasta.aln')
nodL_mismatch = hammingCompareLoci('Bradyrhizobium/Dataset/Complete/withDuplicate/SYM/nodL.combined.duplicate.fasta.aln')
nodZ_mismatch = hammingCompareLoci('Bradyrhizobium/Dataset/Complete/withDuplicate/SYM/nodZ.combined.duplicate.fasta.aln')
        

## List of repeated sequences in the meta file

In [120]:
meta726 = pd.read_csv('../Metadata_Hollo.Geno.726.csv', encoding = 'latin1')

In [127]:
ngs = list(meta726[meta726['Source'] == 'Genomes']['Strain'])

In [161]:
print(nodL_mismatch)

for i in recA_mismatch.keys():
    if i not in ngs:
        print(i)

{'11LoS29_1': 2, '11LoS14_3': 2}



#### recA
05LoS1_7: 1, check SNPs
05LoS3_4: 26, flanking missing, check SNPs
05LoS14_6: 24, check SNPs
05LoS22_10: 18, check SNPs
05LoS23_11: 21, check SNPs
09LoS36_1: 27, check SNPs
09LoS36_3: 23, check SNPs
09LoS36_4: 35, flanking missing, check SNPs
09LoS40_3: 3, check SNPs
09LoS42_2: 1, check SNPs
11LoS8_1: 18, check SNPs
11LoS14_3: 27, flanking missing, check SNPs
11LoS15_1: 4, check SNPs
11LoS17_12: 28, check SNPs
11LoS20_1: 31, check SNPs
11LoS20_4: 108, flanking missing, check SNPs
11LoS21_3: 36, check SNPs
11LoS28_4: 12, check SNPs
11LoS29_1: 36, check SNPs
11LoS33_1: 33, flanking missing, check SNPs
11LoS33_4: 22, flanking missing, check SNPs
11LoS33_5: 31, flanking missing, check SNPs
11LoS31_1: 31, flanking missing, check SNPs
11LoS31_2: 5, flanking missing, check SNPs
11LoS33_5: keep longer
11LoS31_2: keep longer
11LoS31_5: keep longer
11LoS32_1: keep longer
12LoS21_12: 6, flanking missing, check SNPs

#### dnaK
05LoS14_6: keep longer
09LoS43_2: 5, check SNPs
11LoS14_3: 29, check SNPs
11LoS20_4: 1, check SNPs
11LoS32_1: 1, check SNPs

#### ITS
11LoS4_3: keep longer
11LoS28_4: 282, flanking missing, check SNPs
11Los29_1: keep longer
11LoS33_1: keep longer
11LoS33_4: keep longer
11LoS33_5: 434, flanking missing, keep longer, check SNPs
11LoS31_1: keep longer
11LoS31_2: keep longer
11LoS31_5: keep longer
11LoS32_1: keep longer




#### glnII
05LoS3_4 : keep longer
05LoS23_10: 33, check SNPs
05LoS23_11: 1, check SNPs
11LoS17_12: 1, check SNPs
11LoS20_4: keep longer
11LoS31_2: keep longer
11LoS31_5: keep longer
09LoS37_1: keep longer



# Create haplotype meta file

In [70]:
import pandas as pd

#init emptyHashDict

def name2dict(namesFile):
    '''
    Input: Mothur namefile in tsv format
    Output: A dictionary mapping isolate with all it\'s identical haplotype isolates
    '''
    nameDict = {}
    with open(namesFile) as f:
        data = f.readlines()
    for i in data:
        hap, mem = i.split('\t')
        mem = mem.strip().split(',')
        nameDict[hap] = mem
    return(nameDict)

def reverseNameDict(nameDict):
    '''
    Input: Output from name2dict
    Output: All isolates from name2dict output mapping to their unique haplotype isolate id
    '''
    revNameDict = {}
    for i in nameDict.keys():
        for j in nameDict[i]:
            revNameDict[j] = i
        #else:
        #   revNameDict[j] = i
    return revNameDict

def annotateHaplotype(prefix, similarity_list):
    '''
    Input: A list of unique haplotype isolate from meta haplotype matrix and a prefix for the loci
    Output: Map the provided list serially adding numbers after prefix
    '''
    count=1
    annotated = {}
    discard = []
    codedList = []
    for i in similarity_list:
        if i not in discard:            
            haplotype = prefix + str(count)
            count += 1
            discard.append(i)
            annotated[i] = haplotype
        else:
            haplotype = annotated[i]
        codedList.append(annotated[i])
    return codedList

#pd.read_csv('Bradyrhizobium/Dataset/Complete/Dataset/chr/recA.567.names', sep='\t', header=None)

In [71]:
recA_names = reverseNameDict(name2dict('Bradyrhizobium/Dataset/Complete/Dataset/chr/recA.567.fasta.filter.names'))
ITS_names = reverseNameDict(name2dict('Bradyrhizobium/Dataset/Complete/Dataset/chr/ITS.567.fasta.filter.names'))
glnII_names = reverseNameDict(name2dict('Bradyrhizobium/Dataset/Complete/Dataset/chr/glnII.567.fasta.filter.names'))
dnaK_names = reverseNameDict(name2dict('Bradyrhizobium/Dataset/Complete/Dataset/chr/dnaK.567.fasta.filter.names'))

nifD_names = reverseNameDict(name2dict('Bradyrhizobium/Dataset/Complete/Dataset/sym/nifD.brady.fasta.filter.names'))
nodA_names = reverseNameDict(name2dict('Bradyrhizobium/Dataset/Complete/Dataset/sym/nodA.brady.fasta.filter.names'))
nodL_names = reverseNameDict(name2dict('Bradyrhizobium/Dataset/Complete/Dataset/sym/nodL.brady.fasta.filter.names'))
nodZ_names = reverseNameDict(name2dict('Bradyrhizobium/Dataset/Complete/Dataset/sym/nodZ.brady.fasta.filter.names'))



In [72]:
haplotypes = {'dnaK_similarity':dnaK_names, 'recA_similarity':recA_names, 'ITS_similarity':ITS_names, 
              'glnII_similarity':glnII_names,'nifD_similarity':nifD_names, 'nodA_similarity':nodA_names, 
              'nodL_similarity':nodL_names, 'nodZ_similarity':nodZ_names}
haplotypes = pd.DataFrame.from_dict(haplotypes)

haplotypes.index.name = 'Isolate'

conservative_haplotypes = haplotypes.dropna()
conservative_haplotypes

,dnaK_similarity,recA_similarity,ITS_similarity,glnII_similarity,nifD_similarity,nodA_similarity,nodL_similarity,nodZ_similarity
Isolate,,,,,,,,
19AcHe316.1.1,19AcHe316.1.1,19AcS397Asp.3.1,11LoS24_1,11LoS6_3,11LoS6_3,19AcGLB10.1.1,05LoS16_8,11LoS7_2
19AcS309.1.1,19AcHe316.1.1,05LoS13_4,11LoS24_1,11LoS6_3,19AcS309.1.1,19AcGLB10.1.1,19AcS309.1.1,19AcS309.1.1
19AcARN268.1.1,19AcHe316.1.1,19AcS397Asp.3.1,11LoS24_1,11LoS6_3,19AcS5.2.1,19AcS398Asp.1.1,19AcARN268.1.1,11LoS6_3
19AcS398Asp.1.1,19AcHe316.1.1,19AcS397Asp.3.1,19AcS398Asp.1.1,11LoS6_3,12LoS5_2,19AcS398Asp.1.1,05LoS22_6,11LoS6_3
19AcS307.2.1,19AcHe316.1.1,05LoS7_9,11LoS24_1,11LoS6_3,19AcS307.2.1,19AcS307.2.1,19AcS5.1.1,11LoS15_2
...,...,...,...,...,...,...,...,...
12LoS3_5,12LoS3_5,05LoS13_4,12LoS3_5,12LoS3_5,12LoS3_5,12LoS3_5,12LoS3_5,12LoS3_5
12LoS3_8,12LoS3_8,19AcS397Asp.3.1,12LoS3_8,12LoS3_8,12LoS1_4,11LoS19_4,12LoS20_5,11LoS19_4
12LoS16_3,12LoS16_3,19AcS397Asp.3.1,11LoS25_1,11LoS11_2,12LoS16_3,12LoS16_3,19AcGR302.5.1,12LoS16_3


In [73]:
annotate_dnaK = annotateHaplotype('K', list(conservative_haplotypes['dnaK_similarity']))
annotate_recA = annotateHaplotype('R', list(conservative_haplotypes['recA_similarity']))
annotate_ITS = annotateHaplotype('I', list(conservative_haplotypes['ITS_similarity']))
annotate_glnII = annotateHaplotype('G', list(conservative_haplotypes['glnII_similarity']))

annotate_nifD = annotateHaplotype('D', list(conservative_haplotypes['nifD_similarity']))
annotate_nodA = annotateHaplotype('A', list(conservative_haplotypes['nodA_similarity']))
annotate_nodL = annotateHaplotype('L', list(conservative_haplotypes['nodL_similarity']))
annotate_nodZ = annotateHaplotype('Z', list(conservative_haplotypes['nodZ_similarity']))            

In [74]:
conservative_haplotypes['dnaK_coded'] = annotate_dnaK
conservative_haplotypes['recA_coded'] = annotate_recA
conservative_haplotypes['ITS_coded'] = annotate_ITS
conservative_haplotypes['glnII_coded'] = annotate_glnII

conservative_haplotypes['nifD_coded'] = annotate_nifD
conservative_haplotypes['nodA_coded'] = annotate_nodA
conservative_haplotypes['nodL_coded'] = annotate_nodL
conservative_haplotypes['nodZ_coded'] = annotate_nodZ

/Users/arafat/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
/Users/arafat/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  
/Users/arafat/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value in

In [75]:
conservative_haplotypes = conservative_haplotypes[['dnaK_similarity', 'dnaK_coded', 'recA_similarity', 
                                                   'recA_coded', 'ITS_similarity', 'ITS_coded', 'glnII_similarity', 
                                                   'glnII_coded', 'nifD_similarity', 'nifD_coded', 'nodA_similarity', 
                                                   'nodA_coded', 'nodL_similarity', 'nodL_coded', 'nodZ_similarity', 
                                                   'nodZ_coded']]

In [76]:
# Need to incorporate site, sampling longitude and latitude coordinates, and host species from meta file
meta = pd.read_csv('../Metadata_Hollo.Geno.726.csv', encoding = 'latin1')
meta = meta.drop_duplicates(subset=['Strain'], keep='first')

In [78]:
merged.meta = pd.merge(conservative_haplotypes, meta, how='left', left_on=['Isolate'], right_on=['Strain'])

NameError: name 'merged' is not defined